In [2]:
import os
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed


In [4]:

# --- Configuration ---
DATA_URLS = {
    'data/train': 'https://www.cs.toronto.edu/~vmnih/data/mass_buildings/train/sat/',
    'data/train_mask': 'https://www.cs.toronto.edu/~vmnih/data/mass_buildings/train/map/',
    'data/val': 'https://www.cs.toronto.edu/~vmnih/data/mass_buildings/valid/sat/',
    'data/val_mask': 'https://www.cs.toronto.edu/~vmnih/data/mass_buildings/valid/map/',
    'data/test': 'https://www.cs.toronto.edu/~vmnih/data/mass_buildings/test/sat/',
    'data/test_mask': 'https://www.cs.toronto.edu/~vmnih/data/mass_buildings/test/map/'
}

def download_file(file_url, local_filepath):
    """Download a single file."""
    try:
        if os.path.exists(local_filepath):
            return f"Skipping {os.path.basename(local_filepath)}, already exists."

        with requests.get(file_url, stream=True, timeout=30) as r:
            r.raise_for_status()
            total_size = int(r.headers.get('content-length', 0))

            with open(local_filepath, 'wb') as f, tqdm(
                total=total_size, unit='iB', unit_scale=True,
                desc=os.path.basename(local_filepath), leave=False
            ) as pbar:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
                    pbar.update(len(chunk))
        return f"Downloaded {os.path.basename(local_filepath)}"
    except Exception as e:
        return f"❌ Error downloading {file_url}: {e}"

def download_files_from_url(url, destination_folder, max_workers=10):
    """Download all TIFF/TIF files from a given URL into a folder using threading."""
    try:
        print(f"\nFetching links from: {url}")
        response = requests.get(url)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')
        links = soup.find_all('a', href=lambda href: href and (href.endswith('.tiff') or href.endswith('.tif')))

        if not links:
            print(f"⚠️ No image files found at {url}")
            return

        os.makedirs(destination_folder, exist_ok=True)
        print(f"Found {len(links)} files. Starting downloads in parallel...")

        # Build list of tasks
        tasks = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            for link in links:
                file_url = urljoin(url, link.get('href'))
                filename = os.path.basename(file_url)
                local_filepath = os.path.join(destination_folder, filename)
                tasks.append(executor.submit(download_file, file_url, local_filepath))

            for future in tqdm(as_completed(tasks), total=len(tasks), desc=f"Folder: {destination_folder}"):
                print(future.result())

    except Exception as e:
        print(f"❌ Error fetching {url}: {e}")

def main():
    print("\n--- Starting Massachusetts Buildings Dataset Download ---")
    for folder, url in DATA_URLS.items():
        download_files_from_url(url, folder, max_workers=15)  # bump workers for faster downloads
    print("\n--- All downloads complete! ---")
    print("Your data is now organized in the 'data/' directory.")

if __name__ == "__main__":
    main()



--- Starting Massachusetts Buildings Dataset Download ---

Fetching links from: https://www.cs.toronto.edu/~vmnih/data/mass_buildings/train/sat/
Found 137 files. Starting downloads in parallel...


Folder: data/train:   0%|          | 0/137 [00:00<?, ?it/s]














































































































































































































































































































































































































































































































































































































































































































































































































































































































































































Downloaded 22678975_15.tiff


Downloaded 22678915_15.tiff


Downloaded 22678960_15.tiff
Downloaded 22679050_15.tiff
Downloaded 22679020_15.tiff


...

Downloaded 22829050_15.tif




































































































































































































































































































Folder: data/test_mask:  90%|█████████ | 9/10 [00:33<00:03,  3.92s/it]









Downloaded 23729035_15.tif


































































































































































































































































































































































































































































































































































































































































































































































































































































































































































Folder: data/test_mask: 100%|██████████| 10/10 [01:24<00:00,  8.46s/it]

Downloaded 23879080_15.tif

--- All downloads complete! ---
Your data is now organized in the 'data/' directory.
